In [ ]:
import os 
os.getcwd()
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os

In [ ]:

# Load the filez
data = loadmat('data.mat')
illumination = loadmat("illumination.mat")
pose=loadmat("pose.mat")
# See all the keys (variables stored in the .mat file)
print(f"data keys are {data.keys()}")
print(f"illumination keys are {illumination.keys()}")
print(f"pose keys are {pose.keys()}")

In [ ]:
data["face"].shape

In [ ]:
n=np.arange(1,201,1)
data_classified = {}
for i in n:
    imgs=[]
    for j in range(3):
        imgs.append(data["face"][:,:,3*i-3+j])
    data_classified[i]=imgs

In [ ]:
len(data_classified[1])

In [ ]:
n_2=np.arange(0,68,1)
n_2p=np.arange(0,13,1)
pose_classified={}
for i in n_2:
    imgs=[]
    for j in n_2p:
        imgs.append(pose["pose"][:,:,j,i])
    pose_classified[i]=imgs

In [ ]:
n_3=np.arange(0,68,1)
n_3i=np.arange(0,21,1)
illum_classified={}
for i in n_3:
    imgs=[]
    for j in n_3i:
        imgs.append(illumination["illum"][:,j,i])
    illum_classified[i]=imgs

# data + labeling

In [ ]:
flattened_data = np.array([data["face"][:, :, i].flatten() for i in range(600)])
person_labels = np.repeat(np.arange(200), 3) #for task1

# for task2:
neutral_indices = list(range(0, 600, 3))
expression_indices = list(range(1, 600, 3))
binary_labels = np.zeros(600, dtype=int)
binary_labels[expression_indices] = 1


# PCA_funciton

In [ ]:
import numpy as np

def compute_pca(data_matrix, num_components=None, variance_threshold=None):
    """
    Perform PCA on a data matrix (samples × features).
    
    Parameters:
        data_matrix: np.ndarray, shape (n_samples, n_features)
        num_components: int or None — number of components to keep
        variance_threshold: float or None — keep components that explain up to this cumulative variance (e.g., 0.95)
    
    Returns:
        pca_result: projected data, shape (n_samples, num_components)
        components: principal components (eigenvectors)
        explained_variance_ratio: array of variance explained by each component
        mean: mean of original data (for inverse transform if needed)
    """
    # Step 1: Center the data
    mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - mean

    # Step 2: Covariance matrix
    cov_matrix = np.cov(centered_data, rowvar=False)

    # Step 3: Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    # Step 4: Sort in descending order
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 5: Compute explained variance
    explained_variance_ratio = eigenvalues / np.sum(eigenvalues)
    cumulative_variance = np.cumsum(explained_variance_ratio)

    # Step 6: Determine number of components
    if variance_threshold is not None:
        num_components = np.argmax(cumulative_variance >= variance_threshold) + 1
        print(f"Using {num_components} components to explain {variance_threshold*100:.1f}% variance.")
    elif num_components is None:
        num_components = data_matrix.shape[1]  # Keep all

    # Step 7: Select components and project
    selected_components = eigenvectors[:, :num_components]
    pca_result = np.dot(centered_data, selected_components)

    return pca_result, selected_components, explained_variance_ratio[:num_components], mean


# MDA Function

In [ ]:
import numpy as np

def compute_mda(data_matrix, labels, num_components=None):
    """
    Perform MDA (also known as LDA) on the given data.

    Parameters:
        data_matrix: np.ndarray of shape (n_samples, n_features)
        labels: array-like of shape (n_samples,)
        num_components: int or None — number of components to retain (must be ≤ n_classes - 1)

    Returns:
        mda_result: Projected data of shape (n_samples, num_components)
        components: Eigenvectors used for projection (n_features, num_components)
        eigenvalues: Corresponding eigenvalues
        overall_mean: Mean of the original data
    """
    n_samples, n_features = data_matrix.shape
    unique_classes = np.unique(labels)
    n_classes = len(unique_classes)

    # Step 1: Center the data
    overall_mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - overall_mean

    # Step 2: Compute class means
    class_means = []
    for c in unique_classes:
        class_data = data_matrix[labels == c]
        class_mean = np.mean(class_data, axis=0)
        class_means.append(class_mean)

    # Step 3: Compute between-class scatter matrix (S_B)
    S_B = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        n_i = np.sum(labels == c)
        mean_diff = (class_means[i] - overall_mean).reshape(-1, 1)
        S_B += (n_i / n_samples) * (mean_diff @ mean_diff.T)

    # Step 4: Compute within-class scatter matrix (S_W)
    S_W = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        class_data = data_matrix[labels == c]
        n_i = class_data.shape[0]
        class_centered = class_data - class_means[i]
        S_W += (n_i / n_samples) * (class_centered.T @ class_centered) / n_i

    # Step 5: Solve generalized eigenvalue problem
    S_W_inv = np.linalg.pinv(S_W)
    eig_matrix = S_W_inv @ S_B
    eigenvalues, eigenvectors = np.linalg.eigh(eig_matrix)

    # Step 6: Sort eigenvectors by eigenvalue magnitude (descending)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 7: Limit number of components
    max_components = n_classes - 1
    if num_components is None or num_components > max_components:
        num_components = max_components
        print(f"Using max possible components for MDA: {num_components}")

    selected_components = eigenvectors[:, :num_components]
    mda_result = centered_data @ selected_components

    return mda_result, selected_components, eigenvalues[:num_components], overall_mean


# data seperation

In [ ]:
def separate_train_test_manual(data, labels, sub_num_total, train_num, task_num, random_state=42):
    np.random.seed(random_state)
    
    if task_num == 1:  # Person identification with all 3 images
        selected_subjects = np.random.choice(sub_num_total, train_num, replace=False)
        train_indices = []
        test_indices = []

        # For each subject, put 2 images in training and 1 in testing
        for s in selected_subjects:
            base = 3 * s
            train_indices.extend([base, base + 1])  # First 2 images to training
            test_indices.append(base + 2)           # Last image to testing

        train_set = data[train_indices]
        train_labels = labels[train_indices]
        test_set = data[test_indices]
        test_labels = labels[test_indices]

    elif task_num == 2:
        # Get indices for neutral and expression images
        neutral_indices = list(range(0, 3 * sub_num_total, 3))     # Images at positions 0, 3, 6, ...
        expression_indices = list(range(1, 3 * sub_num_total, 3))  # Images at positions 1, 4, 7, ...
        
        # Create binary labels (0 for neutral, 1 for expression)
        neutral_labels = np.zeros(len(neutral_indices), dtype=int)
        expression_labels = np.ones(len(expression_indices), dtype=int)
        
        # Shuffle each set of indices separately
        np.random.shuffle(neutral_indices)
        np.random.shuffle(expression_indices)
        
        # Split each class into training (80%) and testing (20%)
        neutral_split = int(len(neutral_indices) * 0.8)
        expression_split = int(len(expression_indices) * 0.8)
        
        # Create training and testing sets for each class
        neutral_train = neutral_indices[:neutral_split]
        neutral_test = neutral_indices[neutral_split:]
        expression_train = expression_indices[:expression_split]
        expression_test = expression_indices[expression_split:]
        
        # Combine indices and labels
        train_indices = np.concatenate([neutral_train, expression_train])
        test_indices = np.concatenate([neutral_test, expression_test])
        
        # Create labels matching the indices
        train_labels = np.concatenate([np.zeros(len(neutral_train)), np.ones(len(expression_train))])
        test_labels = np.concatenate([np.zeros(len(neutral_test)), np.ones(len(expression_test))])
        
        # Extract the data
        train_set = data[train_indices]
        test_set = data[test_indices]
    
    else:
        raise ValueError("❌ Invalid task_num. Use 1 for person ID or 2 for expression classification.")
    return train_set, train_labels, test_set, test_labels


## bayes classifier

In [ ]:
import numpy as np

def bayes_classifier(train_set, train_labels, test_set, test_labels, num_classes):
    """
    Implements Bayes classifier with Gaussian assumption (QDA-style).
    Parameters:
        train_set: np.ndarray, shape (n_train_samples, n_features)
        train_labels: np.ndarray, shape (n_train_samples,)
        test_set: np.ndarray, shape (n_test_samples, n_features)
        test_labels: np.ndarray, shape (n_test_samples,)
        num_classes: int — number of classes (2 for expression task)
    Returns:
        y_pred: predicted class labels
        accuracy: classification accuracy in percentage
    """
    n_features = train_set.shape[1]
    class_means = {}  # Use dictionary instead of array
    class_covs = {}   # Use dictionary instead of array
    
    # Step 1: Compute MLE of mean and covariance for each class
    for c in range(num_classes):
        class_data = train_set[train_labels == c]
        
        # Check if there are any samples for this class
        if len(class_data) > 0:
            class_means[c] = np.mean(class_data, axis=0)
            
            sigma = np.zeros((n_features, n_features))
            for sample in class_data:
                diff = sample - class_means[c]
                sigma += np.outer(diff, diff)
            class_covs[c] = (sigma / len(class_data)) + np.eye(n_features) * 0.92  # Regularization
        else:
            print(f"Warning: Class {c} not found in training data")
    
    # Step 2: Evaluate discriminant function for each test point
    valid_classes = list(class_means.keys())
    y_pred = np.zeros(len(test_set), dtype=int)
    
    for i, x in enumerate(test_set):
        g = {}  # Use dictionary to store discriminant values
        
        for c in valid_classes:
            cov_inv = np.linalg.inv(class_covs[c])
            cov_det = np.linalg.det(class_covs[c])
            mu = class_means[c]
            
            quad_term = -0.5 * np.dot(np.dot(x, cov_inv), x)
            linear_term = np.dot(mu, np.dot(cov_inv, x))
            const_term = -0.5 * np.dot(np.dot(mu, cov_inv), mu) - 0.5 * np.log(cov_det)
            
            g[c] = quad_term + linear_term + const_term
        
        # Handle the case where no valid classes exist
        if g:
            y_pred[i] = max(g, key=g.get)
        else:
            y_pred[i] = 0  # Default prediction if no valid classes
    
    accuracy = np.mean(y_pred == test_labels) * 100
    accuracy2 = (sum(y_pred == test_labels)/len(test_labels))*100
    accuracy3 = (len(y_pred == test_labels)/len(test_labels))*100
    return y_pred, accuracy, accuracy2, accuracy3

In [ ]:
import matplotlib.pyplot as plt

def plot_bayes_output_vs_labels(y_pred, y_true, task_num=1):
    plt.figure(figsize=(12, 4))

    plt.plot(y_pred, 'o', label="Bayes output", linewidth=2.8, color='blue')
    plt.plot(y_true, '*', label="Labels", linewidth=1.0, color='red')

    plt.title("Classifying Neutral and Expression using Bayes classifier on dataset DATA")
    plt.xlabel("Test sample index")
    plt.ylabel("Class label")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# import matplotlib.pyplot as plt
import numpy as np

def plot_classification_results(y_pred, y_true, task_num=1):
    # Calculate accuracy
    accuracy = np.mean(y_pred == y_true) * 100
    
    # Create a figure
    plt.figure(figsize=(12, 6))
    
    # Create an array to represent if predictions were correct
    correct = y_pred == y_true
    incorrect = ~correct
    
    # Create indices for x-axis
    x = np.arange(len(y_true))
    
    # Plot correctly classified points at y=1
    plt.scatter(x[correct], np.ones_like(x[correct]), color='green', marker='o', s=80, 
                label=f'Correctly classified ({sum(correct)} samples)', alpha=0.7)
    
    # Plot incorrectly classified points at y=2
    plt.scatter(x[incorrect], np.ones_like(x[incorrect])*2, color='red', marker='x', s=80, 
                label=f'Incorrectly classified ({sum(incorrect)} samples)', alpha=0.7)
    
    # Add title and labels
    plt.title(f"Classification Results (Accuracy: {accuracy:.2f}%)")
    plt.xlabel("Test sample index")
    plt.ylabel("Classification result")
    
    # Set y-ticks to 1 and 2 with custom labels
    plt.yticks([1, 2], ['Correct', 'Incorrect'])
    
    # Set y-limits with some padding
    plt.ylim(0.5, 2.5)
    
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Task 1

In [ ]:
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels, 
    sub_num_total=200,
    train_num=200,
    task_num=1
)

In [ ]:
pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=139)

# 4. Project test set using the same PCA basis
centered_test = test_set - mean_vec
pca_result_test = np.dot(centered_test, components)

In [ ]:
mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels, num_components=199)

# 4. Project test set using the same MDA basis
centered_test = test_set - mean_vec
mda_result_test = np.dot(centered_test, components)

In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(train_set, train_labels, test_set, test_labels, num_classes=200)
print(f"✅ Expression Classification Accuracy (Bayes): {accuracy:.2f}%")


In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(pca_result_train, train_labels, pca_result_test, test_labels, num_classes=200)
print(f"✅ Expression Classification Accuracy (Bayes, PCA): {accuracy:.2f}%")


In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(mda_result_train, train_labels, mda_result_test, test_labels, num_classes=200)
print(f"✅ Expression Classification Accuracy (Bayes, MDA): {accuracy:.2f}%")


In [ ]:
plot_bayes_output_vs_labels(y_pred, test_labels, task_num=1)
plot_classification_results(y_pred, test_labels, task_num=1)

In [ ]:
import matplotlib.pyplot as plt

train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels, 
    sub_num_total=200,
    train_num=200,
    task_num=1
)

# Define component ranges
pca_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139]
mda_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139, 150, 170, 199]

# Store results
pca_accuracies = {}
mda_accuracies = {}

# Run PCA experiments
for num_components in pca_components_list:
    pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=num_components)
    centered_test = test_set - mean_vec
    pca_result_test = np.dot(centered_test, components)
    
    _, accuracy, _, _ = bayes_classifier(pca_result_train, train_labels, pca_result_test, test_labels, num_classes=200)
    pca_accuracies[num_components] = accuracy
    print(f"PCA {num_components} components → Accuracy: {accuracy:.2f}%")

# Run MDA experiments
for num_components in mda_components_list:
    mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels, num_components=num_components)
    centered_test = test_set - mean_vec
    mda_result_test = np.dot(centered_test, components)
    
    _, accuracy, _, _ = bayes_classifier(mda_result_train, train_labels, mda_result_test, test_labels, num_classes=200)
    mda_accuracies[num_components] = accuracy
    print(f"MDA {num_components} components → Accuracy: {accuracy:.2f}%")

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(list(pca_accuracies.keys()), list(pca_accuracies.values()), marker='o', label='PCA + Bayes')
plt.plot(list(mda_accuracies.keys()), list(mda_accuracies.values()), marker='s', label='MDA + Bayes')

plt.title('Classification Accuracy vs. Number of Components')
plt.xlabel('Number of Components')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
plt.plot(list(pca_accuracies.keys()), list(pca_accuracies.values()), marker='o', label='PCA + Bayes')
plt.plot(list(mda_accuracies.keys()), list(mda_accuracies.values()), marker='s', label='MDA + Bayes')
plt.axhline(64.00, color ='red',linestyle='--', label=f'normal bayes: 64.00%' )

plt.title('Classification Accuracy task 1 vs. Number of Components')
plt.xlabel('Number of Components')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Task 2

In [ ]:
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels, 
    sub_num_total=200,
    train_num=200,  # Use all subjects
    task_num=2
)

len(test_set), len(train_set)

In [ ]:
pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=139)

# 4. Project test set using the same PCA basis
centered_test = test_set - mean_vec
pca_result_test = np.dot(centered_test, components)

In [ ]:
mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels, num_components=1)

# 4. Project test set using the same MDA basis
centered_test = test_set - mean_vec
mda_result_test = np.dot(centered_test, components)

In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(train_set, train_labels, test_set, test_labels, num_classes=2)
print(f"✅ Expression Classification Accuracy (Bayes): {accuracy:.2f}%")


In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(pca_result_train, train_labels, pca_result_test, test_labels, num_classes=2)
print(f"✅ Expression Classification Accuracy (Bayes, PCA): {accuracy:.2f}%")


In [ ]:
plot_bayes_output_vs_labels(y_pred, test_labels, task_num=1)
plot_classification_results(y_pred, test_labels, task_num=1)

In [ ]:
components_number=[1,2,3,10,50,80,100,120,139]
pca_accuracies={}

for i in components_number: 
    
    pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=i)

    # 4. Project test set using the same PCA basis
    centered_test = test_set - mean_vec
    pca_result_test = np.dot(centered_test, components)
    
    # Step 4: Run Bayes classifier
    y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(pca_result_train, train_labels, pca_result_test, test_labels, num_classes=2)
    print(f"✅ Expression Classification Accuracy (Bayes, PCA): {accuracy:.2f}%")
    pca_accuracies[i]=accuracy

In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(mda_result_train, train_labels, mda_result_test, test_labels, num_classes=2)
print(f"✅ Expression Classification Accuracy (Bayes, MDA): {accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Given PCA accuracies from your printouts
components_number = [1, 2, 3, 10, 50, 80, 100, 120, 139]
pca_accuracies = {
    1: 62.5,
    2: 58.75,
    3: 75.0,
    10: 80.0,
    50: 78.75,
    80: 81.25,
    100: 81.25,
    120 :80.00, 
    139: 78.75
}

# MDA accuracy
mda_accuracy = 92.5

# regular_bayes
normal_bayes = 78.75

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(components_number, [pca_accuracies[k] for k in components_number], marker='o', label='Bayes (PCA)')
plt.axhline(mda_accuracy, color='red', linestyle='--', label=f'Bayes (MDA): {mda_accuracy:.2f}%')
plt.axhline(normal_bayes, color ='orange',linestyle='--', label=f'normal bayes: {mda_accuracy:.2f}%' )

plt.title('Expression Classification Accuracy task 2')
plt.xlabel('Number of PCA Components')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
